# 02 - Silver: Limpeza e Validação com PySpark

Este notebook executa apenas os itens solicitados para a camada Silver:

- limpeza de dados;
- tratamento de valores ausentes;
- padronização de nomes e tipos;
- validação de consistência;
- normalização de chaves.

**Arquitetura Unity Catalog:**
- **Entrada**: Lê da camada bronze em `workspace.bronze.*`
- **Saída**: Salva na camada silver em `workspace.silver.*`
- Todas as bases são persistidas como tabelas Delta gerenciadas no Unity Catalog

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("tech-challenge-silver").getOrCreate()

# Configuração Unity Catalog
BRONZE_CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_CATALOG = "workspace"
SILVER_SCHEMA = "silver"  # Schema dedicado para camada Silver

print(f"Entrada: {BRONZE_CATALOG}.{BRONZE_SCHEMA}")
print(f"Saída: {SILVER_CATALOG}.{SILVER_SCHEMA}")

# Criar schema silver se não existir
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.{SILVER_SCHEMA}")
print(f"✓ Schema {SILVER_CATALOG}.{SILVER_SCHEMA} pronto")

Entrada: workspace.bronze
Saída: workspace.silver
✓ Schema workspace.silver pronto


In [0]:
def ler_tabela_bronze(nome_tabela):
    """Lê uma tabela da camada bronze do Unity Catalog"""
    return spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.{nome_tabela}")


def limpar_texto(coluna):
    return F.trim(F.col(coluna).cast("string"))


def para_inteiro(coluna):
    valor = limpar_texto(coluna)
    return F.when(valor == "", None).otherwise(valor.cast("int"))


def para_decimal(coluna):
    valor = F.regexp_replace(limpar_texto(coluna), ",", ".")
    valor = F.regexp_replace(valor, ">", "")
    valor = F.trim(valor)
    return F.when(valor == "", None).otherwise(valor.cast("double"))


def normalizar_rede(coluna):
    valor = F.lower(limpar_texto(coluna))
    return (
        F.when(valor == "0", "total")
        .when(valor == "2", "estadual")
        .when(valor == "3", "municipal")
        .when(valor == "5", "publica")
        .when(valor.isin("pública", "p�blica", "publica"), "publica")
        .when(valor == "municipal", "municipal")
        .when(valor == "estadual", "estadual")
        .when(valor == "total", "total")
        .otherwise(valor)
    )


def salvar_tabela_silver(df, nome_tabela):
    """Salva DataFrame como tabela Delta gerenciada no Unity Catalog"""
    tabela_completa = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.{nome_tabela}"
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_completa)
    count = df.count()
    print(f"✓ Tabela Silver criada: {tabela_completa} ({count:,} registros)")


def perfil(df, nome):
    total = df.count()
    ausentes = {
        coluna: df.filter(F.col(coluna).isNull() | (F.col(coluna).cast("string") == "")).count()
        for coluna in df.columns
    }
    return {
        "base": nome,
        "linhas": total,
        "colunas": df.columns,
        "valores_ausentes": ausentes,
    }

## Leitura Das Bases

As entradas são as tabelas da camada bronze no Unity Catalog (`workspace.bronze.*`). Nenhuma integração entre bases é feita nesta etapa.

In [0]:
# Leitura das tabelas bronze do Unity Catalog
indicador_uf_raw = ler_tabela_bronze("indicador_alfabetizacao_uf")
indicador_municipio_raw = ler_tabela_bronze("indicador_alfabetizacao_municipio")
meta_brasil_raw = ler_tabela_bronze("meta_alfabetizacao_brasil")
meta_uf_raw = ler_tabela_bronze("meta_alfabetizacao_uf")
meta_municipio_raw = ler_tabela_bronze("meta_alfabetizacao_municipio")
microdados_raw = ler_tabela_bronze("microdados_alunos")

# Esta base veio de planilha com cabeçalhos extras que foram removidos na bronze
resultados_uf_2025_raw = ler_tabela_bronze("resultados_e_metas_ufs_2025")

In [0]:
nivel_cols = [f"proporcao_aluno_nivel_{i}" for i in range(9)]

indicador_uf = indicador_uf_raw.select(
    para_inteiro("ano").alias("ano"),
    F.upper(limpar_texto("sigla_uf")).alias("sigla_uf"),
    para_inteiro("serie").alias("serie"),
    limpar_texto("rede").alias("rede_codigo"),
    normalizar_rede("rede").alias("rede"),
    para_decimal("taxa_alfabetizacao").alias("taxa_alfabetizacao"),
    para_decimal("media_portugues").alias("media_portugues"),
    *[para_decimal(coluna).alias(coluna) for coluna in nivel_cols],
)

indicador_municipio = indicador_municipio_raw.select(
    para_inteiro("ano").alias("ano"),
    limpar_texto("id_municipio").alias("id_municipio"),
    para_inteiro("serie").alias("serie"),
    limpar_texto("rede").alias("rede_codigo"),
    normalizar_rede("rede").alias("rede"),
    para_decimal("taxa_alfabetizacao").alias("taxa_alfabetizacao"),
    para_decimal("media_portugues").alias("media_portugues"),
    *[para_decimal(coluna).alias(coluna) for coluna in nivel_cols],
)

indicador_uf.show(5)
indicador_municipio.show(5)

+----+--------+-----+-----------+---------+------------------+---------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+
| ano|sigla_uf|serie|rede_codigo|     rede|taxa_alfabetizacao|media_portugues|proporcao_aluno_nivel_0|proporcao_aluno_nivel_1|proporcao_aluno_nivel_2|proporcao_aluno_nivel_3|proporcao_aluno_nivel_4|proporcao_aluno_nivel_5|proporcao_aluno_nivel_6|proporcao_aluno_nivel_7|proporcao_aluno_nivel_8|
+----+--------+-----+-----------+---------+------------------+---------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+
|2023|      AM|    2|          3|municipal|              49.2|       733.6637|                   NULL|             

In [0]:
meta_brasil = meta_brasil_raw.select(
    para_inteiro("ano").alias("ano"),
    normalizar_rede("rede").alias("rede"),
    para_decimal("taxa_alfabetizacao").alias("taxa_alfabetizacao"),
    para_decimal("meta_alfabetizacao_2024").alias("meta_2024"),
    para_decimal("meta_alfabetizacao_2025").alias("meta_2025"),
    para_decimal("meta_alfabetizacao_2026").alias("meta_2026"),
    para_decimal("meta_alfabetizacao_2027").alias("meta_2027"),
    para_decimal("meta_alfabetizacao_2028").alias("meta_2028"),
    para_decimal("meta_alfabetizacao_2029").alias("meta_2029"),
    para_decimal("meta_alfabetizacao_2030").alias("meta_2030"),
    para_decimal("percentual_participacao").alias("percentual_participacao"),
)

meta_uf = meta_uf_raw.select(
    para_inteiro("ano").alias("ano"),
    F.upper(limpar_texto("sigla_uf")).alias("sigla_uf"),
    normalizar_rede("rede").alias("rede"),
    para_decimal("taxa_alfabetizacao").alias("taxa_alfabetizacao"),
    para_decimal("meta_alfabetizacao_2024").alias("meta_2024"),
    para_decimal("meta_alfabetizacao_2025").alias("meta_2025"),
    para_decimal("meta_alfabetizacao_2026").alias("meta_2026"),
    para_decimal("meta_alfabetizacao_2027").alias("meta_2027"),
    para_decimal("meta_alfabetizacao_2028").alias("meta_2028"),
    para_decimal("meta_alfabetizacao_2029").alias("meta_2029"),
    para_decimal("meta_alfabetizacao_2030").alias("meta_2030"),
    para_decimal("percentual_participacao").alias("percentual_participacao"),
)

meta_municipio = meta_municipio_raw.select(
    para_inteiro("ano").alias("ano"),
    limpar_texto("id_municipio").alias("id_municipio"),
    normalizar_rede("rede").alias("rede"),
    para_decimal("taxa_alfabetizacao").alias("taxa_alfabetizacao"),
    para_decimal("meta_alfabetizacao_2024").alias("meta_2024"),
    para_decimal("meta_alfabetizacao_2025").alias("meta_2025"),
    para_decimal("meta_alfabetizacao_2026").alias("meta_2026"),
    para_decimal("meta_alfabetizacao_2027").alias("meta_2027"),
    para_decimal("meta_alfabetizacao_2028").alias("meta_2028"),
    para_decimal("meta_alfabetizacao_2029").alias("meta_2029"),
    para_decimal("meta_alfabetizacao_2030").alias("meta_2030"),
    para_inteiro("nivel_alfabetizacao").alias("nivel_alfabetizacao"),
    para_decimal("percentual_participacao").alias("percentual_participacao"),
)

meta_brasil.show(5)
meta_uf.show(5)
meta_municipio.show(5)

+----+-------+------------------+---------+---------+---------+---------+---------+---------+---------+-----------------------+
| ano|   rede|taxa_alfabetizacao|meta_2024|meta_2025|meta_2026|meta_2027|meta_2028|meta_2029|meta_2030|percentual_participacao|
+----+-------+------------------+---------+---------+---------+---------+---------+---------+---------+-----------------------+
|2025|publica|              66.0|     60.0|     64.0|     67.0|     71.0|     74.0|     77.0|     80.0|                   88.0|
|2024|publica|              59.2|     59.9|    63.77|    67.47|    70.97|    74.23|    77.24|     80.0|                  87.37|
|2023|publica|              55.9|     59.9|    63.77|    67.47|    70.97|    74.23|    77.24|     80.0|                   86.0|
+----+-------+------------------+---------+---------+---------+---------+---------+---------+---------+-----------------------+

+----+--------+-------+------------------+---------+---------+---------+---------+---------+---------+-

In [0]:
microdados_alunos = microdados_raw.select(
    para_inteiro("ano").alias("ano"),
    limpar_texto("id_municipio").alias("id_municipio"),
    limpar_texto("id_escola").alias("id_escola"),
    limpar_texto("id_aluno").alias("id_aluno"),
    para_inteiro("caderno").alias("caderno"),
    para_inteiro("serie").alias("serie"),
    limpar_texto("rede").alias("rede_codigo"),
    normalizar_rede("rede").alias("rede"),
    para_inteiro("presenca").alias("presenca"),
    para_inteiro("preenchimento_caderno").alias("preenchimento_caderno"),
    para_inteiro("alfabetizado").alias("alfabetizado"),
    para_decimal("proficiencia").alias("proficiencia"),
    para_decimal("peso_aluno").alias("peso_aluno"),
)

microdados_alunos.show(5)

+----+------------+---------+--------+-------+-----+-----------+---------+--------+---------------------+------------+------------+----------+
| ano|id_municipio|id_escola|id_aluno|caderno|serie|rede_codigo|     rede|presenca|preenchimento_caderno|alfabetizado|proficiencia|peso_aluno|
+----+------------+---------+--------+-------+-----+-----------+---------+--------+---------------------+------------+------------+----------+
|2023|     1701903| 60004032|17013597|      1|    2|          3|municipal|       0|                    0|           0|        NULL|      NULL|
|2023|     2109502| 60005472|21061500|      1|    2|          3|municipal|       0|                    0|           0|        NULL|      NULL|
|2023|     2100436| 60006118|21043312|      1|    2|          3|municipal|       0|                    0|           0|        NULL|      NULL|
|2023|     2209856| 60006816|22035405|      1|    2|          3|municipal|       0|                    0|           0|        NULL|      NULL|

In [0]:
# Filtrar cabeçalho duplicado e linhas inválidas
resultados_uf_2025 = (
    resultados_uf_2025_raw
    .filter(
        (limpar_texto("ano_da_avaliacao") != "ANO") &  # Remove cabeçalho duplicado
        (limpar_texto("ano_da_avaliacao") == "2025") &  # Apenas ano 2025
        (limpar_texto("codigo_uf").isNotNull())  # Remove linha "Brasil" agregada
    )
    .select(
        para_inteiro("ano_da_avaliacao").alias("ano"),
        para_inteiro("codigo_uf").alias("codigo_uf"),
        F.upper(F.regexp_replace(limpar_texto("sigla_uf"), "\\*", "")).alias("sigla_uf"),
        limpar_texto("nome_uf").alias("nome_uf"),
        normalizar_rede("rede").alias("rede"),
        para_decimal("percentual_de_alunos_alfabetizados_sistemas_estaduais_de_avaliacao_2023").alias("resultado_2023"),
        para_decimal("percentual_de_alunos_alfabetizados_sistemas_estaduais_de_avaliacao_2024").alias("resultado_2024"),
        para_decimal("percentual_de_alunos_alfabetizados_sistemas_estaduais_de_avaliacao_2025").alias("resultado_2025"),
        para_decimal("meta_2024_2").alias("meta_2024"),
        para_decimal("meta_2025").alias("meta_2025"),
        para_decimal("meta_2026").alias("meta_2026"),
        para_decimal("meta_2027").alias("meta_2027"),
        para_decimal("meta_2028").alias("meta_2028"),
        para_decimal("meta_2029").alias("meta_2029"),
        para_decimal("meta_2030").alias("meta_2030"),
        para_decimal("percentual_de_participacao").alias("percentual_participacao"),
    )
)

resultados_uf_2025.show(5)

+----+---------+--------+--------+-------+--------------+--------------+--------------+---------+---------+---------+---------+---------+---------+---------+-----------------------+
| ano|codigo_uf|sigla_uf| nome_uf|   rede|resultado_2023|resultado_2024|resultado_2025|meta_2024|meta_2025|meta_2026|meta_2027|meta_2028|meta_2029|meta_2030|percentual_participacao|
+----+---------+--------+--------+-------+--------------+--------------+--------------+---------+---------+---------+---------+---------+---------+---------+-----------------------+
|2025|       12|      AC|    Acre|publica|          NULL|          51.0|          68.0|     NULL|     57.0|     62.0|     67.0|     72.0|     76.0|     80.0|                   82.0|
|2025|       27|      AL| Alagoas|publica|          44.0|          49.0|          64.0|     50.0|     56.0|     61.0|     67.0|     72.0|     76.0|     80.0|                   94.0|
|2025|       13|      AM|Amazonas|publica|          52.0|          49.0|          57.0|   

## Validação de Consistência

Esta seção implementa validações de qualidade de dados conforme requisitos da camada Silver:

1. **Verificação de Duplicados**: Identifica registros duplicados em chaves primárias
2. **Validação de Intervalos**: Verifica se valores numéricos estão em intervalos válidos
3. **Validação de Códigos**: Verifica se códigos de UF/município são válidos
4. **Integridade Referencial**: Valida relações entre tabelas
5. **Relatório de Valores Ausentes**: Documenta campos com dados faltantes

In [0]:
# Funções de Validação

def validar_duplicados(df, colunas_chave, nome_tabela):
    """
    Verifica duplicados nas colunas de chave primária
    Retorna: (tem_duplicados, count_duplicados, amostra_duplicados)
    """
    total = df.count()
    distintos = df.select(colunas_chave).distinct().count()
    duplicados = total - distintos
    
    resultado = {
        "tabela": nome_tabela,
        "total_registros": total,
        "registros_unicos": distintos,
        "duplicados": duplicados,
        "tem_duplicados": duplicados > 0,
        "percentual_duplicados": round((duplicados / total * 100) if total > 0 else 0, 2)
    }
    
    if duplicados > 0:
        # Mostra amostra dos duplicados
        df_duplicados = (
            df.groupBy(colunas_chave)
            .count()
            .filter(F.col("count") > 1)
            .orderBy(F.desc("count"))
            .limit(10)
        )
        resultado["amostra"] = df_duplicados
    
    return resultado


def validar_intervalo(df, coluna, min_val, max_val, nome_tabela):
    """
    Valida se valores estão dentro de um intervalo esperado
    """
    total = df.filter(F.col(coluna).isNotNull()).count()
    
    if total == 0:
        return {
            "tabela": nome_tabela,
            "coluna": coluna,
            "valido": True,
            "total_valores": 0,
            "fora_intervalo": 0,
            "mensagem": "Coluna sem valores não-nulos"
        }
    
    fora = df.filter(
        F.col(coluna).isNotNull() & 
        ((F.col(coluna) < min_val) | (F.col(coluna) > max_val))
    ).count()
    
    return {
        "tabela": nome_tabela,
        "coluna": coluna,
        "intervalo": f"[{min_val}, {max_val}]",
        "total_valores": total,
        "fora_intervalo": fora,
        "percentual_invalido": round((fora / total * 100) if total > 0 else 0, 2),
        "valido": fora == 0
    }


def validar_codigos_uf(df, coluna_uf, nome_tabela):
    """
    Valida se códigos de UF são válidos (27 UFs brasileiras)
    """
    ufs_validas = [
        "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA",
        "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI", "RJ", "RN",
        "RS", "RO", "RR", "SC", "SP", "SE", "TO"
    ]
    
    total = df.filter(F.col(coluna_uf).isNotNull()).count()
    invalidos = df.filter(
        F.col(coluna_uf).isNotNull() & 
        ~F.col(coluna_uf).isin(ufs_validas)
    ).count()
    
    resultado = {
        "tabela": nome_tabela,
        "coluna": coluna_uf,
        "total_registros": total,
        "codigos_invalidos": invalidos,
        "percentual_invalido": round((invalidos / total * 100) if total > 0 else 0, 2),
        "valido": invalidos == 0
    }
    
    if invalidos > 0:
        invalidos_sample = (
            df.filter(~F.col(coluna_uf).isin(ufs_validas))
            .select(coluna_uf)
            .distinct()
            .limit(20)
        )
        resultado["amostra_invalidos"] = invalidos_sample
    
    return resultado


def validar_anos(df, coluna_ano, nome_tabela, anos_esperados=[2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030]):
    """
    Valida se anos estão dentro do conjunto esperado
    """
    total = df.filter(F.col(coluna_ano).isNotNull()).count()
    invalidos = df.filter(
        F.col(coluna_ano).isNotNull() & 
        ~F.col(coluna_ano).isin(anos_esperados)
    ).count()
    
    return {
        "tabela": nome_tabela,
        "coluna": coluna_ano,
        "total_registros": total,
        "anos_invalidos": invalidos,
        "anos_esperados": anos_esperados,
        "percentual_invalido": round((invalidos / total * 100) if total > 0 else 0, 2),
        "valido": invalidos == 0
    }


def relatorio_valores_ausentes(df, nome_tabela):
    """
    Gera relatório detalhado de valores ausentes por coluna
    """
    total_registros = df.count()
    colunas = df.columns
    
    relatorio = []
    for coluna in colunas:
        nulos = df.filter(F.col(coluna).isNull()).count()
        percentual = round((nulos / total_registros * 100) if total_registros > 0 else 0, 2)
        
        relatorio.append({
            "tabela": nome_tabela,
            "coluna": coluna,
            "valores_ausentes": nulos,
            "total_registros": total_registros,
            "percentual_ausente": percentual,
            "completude": round(100 - percentual, 2)
        })
    
    return relatorio


def validar_integridade_referencial(df_origem, coluna_origem, df_destino, coluna_destino, 
                                     nome_origem, nome_destino):
    """
    Valida integridade referencial entre duas tabelas
    Verifica se todos os valores em df_origem existem em df_destino
    """
    valores_origem = df_origem.select(coluna_origem).filter(F.col(coluna_origem).isNotNull()).distinct()
    valores_destino = df_destino.select(coluna_destino).filter(F.col(coluna_destino).isNotNull()).distinct()
    
    # Left anti join para encontrar valores que não existem no destino
    orfaos = valores_origem.join(
        valores_destino,
        valores_origem[coluna_origem] == valores_destino[coluna_destino],
        "left_anti"
    )
    
    count_orfaos = orfaos.count()
    
    return {
        "origem": nome_origem,
        "coluna_origem": coluna_origem,
        "destino": nome_destino,
        "coluna_destino": coluna_destino,
        "valores_orfaos": count_orfaos,
        "valido": count_orfaos == 0,
        "amostra_orfaos": orfaos.limit(20) if count_orfaos > 0 else None
    }


print("✓ Funções de validação carregadas")

✓ Funções de validação carregadas


In [0]:
print("="*80)
print("EXECUTANDO VALIDAÇÕES DE QUALIDADE - CAMADA SILVER")
print("="*80)

# Dicionário para armazenar todos os resultados
validacoes = {
    "duplicados": [],
    "intervalos": [],
    "codigos": [],
    "anos": [],
    "integridade": [],
    "valores_ausentes": []
}

# ============================================================================
# 1. VERIFICAÇÃO DE DUPLICADOS
# ============================================================================
print("\n1. Verificando duplicados...")

# Indicador UF - chave: ano + sigla_uf + serie + rede
validacoes["duplicados"].append(
    validar_duplicados(indicador_uf, ["ano", "sigla_uf", "serie", "rede"], "indicador_alfabetizacao_uf")
)

# Indicador Município - chave: ano + id_municipio + serie + rede
validacoes["duplicados"].append(
    validar_duplicados(indicador_municipio, ["ano", "id_municipio", "serie", "rede"], "indicador_alfabetizacao_municipio")
)

# Meta Brasil - chave: ano + rede
validacoes["duplicados"].append(
    validar_duplicados(meta_brasil, ["ano", "rede"], "meta_alfabetizacao_brasil")
)

# Meta UF - chave: ano + sigla_uf + rede
validacoes["duplicados"].append(
    validar_duplicados(meta_uf, ["ano", "sigla_uf", "rede"], "meta_alfabetizacao_uf")
)

# Meta Município - chave: ano + id_municipio + rede
validacoes["duplicados"].append(
    validar_duplicados(meta_municipio, ["ano", "id_municipio", "rede"], "meta_alfabetizacao_municipio")
)

# Microdados - chave: id_aluno (cada aluno aparece uma vez)
validacoes["duplicados"].append(
    validar_duplicados(microdados_alunos, ["id_aluno"], "microdados_alunos")
)

# Resultados UF 2025 - chave: codigo_uf + rede
validacoes["duplicados"].append(
    validar_duplicados(resultados_uf_2025, ["codigo_uf", "rede"], "resultados_e_metas_ufs_2025")
)

print("✓ Duplicados verificados")

# ============================================================================
# 2. VALIDAÇÃO DE INTERVALOS
# ============================================================================
print("\n2. Validando intervalos numéricos...")

# Taxas de alfabetização devem estar entre 0 e 100
for tabela_nome, df in [
    ("indicador_alfabetizacao_uf", indicador_uf),
    ("indicador_alfabetizacao_municipio", indicador_municipio),
    ("meta_alfabetizacao_brasil", meta_brasil),
    ("meta_alfabetizacao_uf", meta_uf),
    ("meta_alfabetizacao_municipio", meta_municipio),
]:
    validacoes["intervalos"].append(
        validar_intervalo(df, "taxa_alfabetizacao", 0, 100, tabela_nome)
    )

# Percentual de participação deve estar entre 0 e 100
for tabela_nome, df in [
    ("meta_alfabetizacao_brasil", meta_brasil),
    ("meta_alfabetizacao_uf", meta_uf),
    ("meta_alfabetizacao_municipio", meta_municipio),
    ("resultados_e_metas_ufs_2025", resultados_uf_2025),
]:
    validacoes["intervalos"].append(
        validar_intervalo(df, "percentual_participacao", 0, 100, tabela_nome)
    )

# Média português deve estar entre 0 e 1000 (escala de proficiência)
for tabela_nome, df in [
    ("indicador_alfabetizacao_uf", indicador_uf),
    ("indicador_alfabetizacao_municipio", indicador_municipio),
]:
    validacoes["intervalos"].append(
        validar_intervalo(df, "media_portugues", 0, 1000, tabela_nome)
    )

# Proporções devem estar entre 0 e 100
for i in range(9):
    coluna = f"proporcao_aluno_nivel_{i}"
    validacoes["intervalos"].append(
        validar_intervalo(indicador_uf, coluna, 0, 100, f"indicador_alfabetizacao_uf.{coluna}")
    )

# Proficiência em microdados deve estar entre 0 e 1000
validacoes["intervalos"].append(
    validar_intervalo(microdados_alunos, "proficiencia", 0, 1000, "microdados_alunos")
)

print("✓ Intervalos validados")

# ============================================================================
# 3. VALIDAÇÃO DE CÓDIGOS DE UF
# ============================================================================
print("\n3. Validando códigos de UF...")

for tabela_nome, df in [
    ("indicador_alfabetizacao_uf", indicador_uf),
    ("meta_alfabetizacao_uf", meta_uf),
    ("resultados_e_metas_ufs_2025", resultados_uf_2025),
]:
    validacoes["codigos"].append(
        validar_codigos_uf(df, "sigla_uf", tabela_nome)
    )

print("✓ Códigos de UF validados")

# ============================================================================
# 4. VALIDAÇÃO DE ANOS
# ============================================================================
print("\n4. Validando anos...")

# Todas as tabelas devem ter anos válidos
for tabela_nome, df in [
    ("indicador_alfabetizacao_uf", indicador_uf),
    ("indicador_alfabetizacao_municipio", indicador_municipio),
    ("meta_alfabetizacao_brasil", meta_brasil),
    ("meta_alfabetizacao_uf", meta_uf),
    ("meta_alfabetizacao_municipio", meta_municipio),
    ("microdados_alunos", microdados_alunos),
    ("resultados_e_metas_ufs_2025", resultados_uf_2025),
]:
    validacoes["anos"].append(
        validar_anos(df, "ano", tabela_nome)
    )

print("✓ Anos validados")

# ============================================================================
# 5. VALIDAÇÃO DE INTEGRIDADE REFERENCIAL
# ============================================================================
print("\n5. Validando integridade referencial...")

# Verifica se UFs em indicadores existem em metas
validacoes["integridade"].append(
    validar_integridade_referencial(
        indicador_uf, "sigla_uf",
        meta_uf, "sigla_uf",
        "indicador_alfabetizacao_uf", "meta_alfabetizacao_uf"
    )
)

# Verifica se municípios em indicadores existem em metas
validacoes["integridade"].append(
    validar_integridade_referencial(
        indicador_municipio, "id_municipio",
        meta_municipio, "id_municipio",
        "indicador_alfabetizacao_municipio", "meta_alfabetizacao_municipio"
    )
)

# Verifica se municípios em microdados existem em indicadores
validacoes["integridade"].append(
    validar_integridade_referencial(
        microdados_alunos, "id_municipio",
        indicador_municipio, "id_municipio",
        "microdados_alunos", "indicador_alfabetizacao_municipio"
    )
)

print("✓ Integridade referencial validada")

# ============================================================================
# 6. RELATÓRIO DE VALORES AUSENTES
# ============================================================================
print("\n6. Gerando relatório de valores ausentes...")

for tabela_nome, df in [
    ("indicador_alfabetizacao_uf", indicador_uf),
    ("indicador_alfabetizacao_municipio", indicador_municipio),
    ("meta_alfabetizacao_brasil", meta_brasil),
    ("meta_alfabetizacao_uf", meta_uf),
    ("meta_alfabetizacao_municipio", meta_municipio),
    ("microdados_alunos", microdados_alunos),
    ("resultados_e_metas_ufs_2025", resultados_uf_2025),
]:
    validacoes["valores_ausentes"].extend(
        relatorio_valores_ausentes(df, tabela_nome)
    )

print("✓ Relatório de valores ausentes gerado")

print("\n" + "="*80)
print("✓ TODAS AS VALIDAÇÕES CONCLUÍDAS")
print("="*80)

EXECUTANDO VALIDAÇÕES DE QUALIDADE - CAMADA SILVER

1. Verificando duplicados...
✓ Duplicados verificados

2. Validando intervalos numéricos...
✓ Intervalos validados

3. Validando códigos de UF...
✓ Códigos de UF validados

4. Validando anos...
✓ Anos validados

5. Validando integridade referencial...
✓ Integridade referencial validada

6. Gerando relatório de valores ausentes...
✓ Relatório de valores ausentes gerado

✓ TODAS AS VALIDAÇÕES CONCLUÍDAS


In [0]:
# ============================================================================
# RESUMO CONSOLIDADO DAS VALIDAÇÕES
# ============================================================================

print("\n" + "="*80)
print("RESUMO DAS VALIDAÇÕES DE QUALIDADE")
print("="*80)

# 1. DUPLICADOS
print("\n1. DUPLICADOS")
print("-" * 80)
for v in validacoes["duplicados"]:
    status = "✗ ENCONTRADOS" if v["tem_duplicados"] else "✓ OK"
    print(f"{status:15} | {v['tabela']:40} | {v['duplicados']:,} duplicados ({v['percentual_duplicados']}%)")
    if v["tem_duplicados"] and "amostra" in v:
        print("  Amostra dos duplicados:")
        v["amostra"].show(5, truncate=False)

# 2. INTERVALOS
print("\n2. INTERVALOS NUMÉRICOS")
print("-" * 80)
problemas_intervalo = [v for v in validacoes["intervalos"] if not v["valido"]]
if problemas_intervalo:
    for v in problemas_intervalo:
        print(f"✗ PROBLEMA | {v['tabela']:40} | {v['coluna']:30}")
        print(f"            Intervalo esperado: {v['intervalo']}")
        print(f"            Valores fora: {v['fora_intervalo']:,} ({v['percentual_invalido']}%)")
else:
    print("✓ Todos os valores numéricos estão dentro dos intervalos esperados")

# 3. CÓDIGOS DE UF
print("\n3. CÓDIGOS DE UF")
print("-" * 80)
for v in validacoes["codigos"]:
    status = "✗ INVÁLIDOS" if not v["valido"] else "✓ OK"
    print(f"{status:15} | {v['tabela']:40} | {v['codigos_invalidos']} inválidos ({v['percentual_invalido']}%)")
    if not v["valido"] and "amostra_invalidos" in v:
        print("  Códigos inválidos encontrados:")
        v["amostra_invalidos"].show(truncate=False)

# 4. ANOS
print("\n4. ANOS")
print("-" * 80)
problemas_ano = [v for v in validacoes["anos"] if not v["valido"]]
if problemas_ano:
    for v in problemas_ano:
        print(f"✗ PROBLEMA | {v['tabela']:40} | {v['anos_invalidos']} anos inválidos ({v['percentual_invalido']}%)")
else:
    print("✓ Todos os anos estão dentro do conjunto esperado [2023-2030]")

# 5. INTEGRIDADE REFERENCIAL
print("\n5. INTEGRIDADE REFERENCIAL")
print("-" * 80)
for v in validacoes["integridade"]:
    status = "✗ QUEBRADA" if not v["valido"] else "✓ OK"
    print(f"{status:15} | {v['origem']:30} → {v['destino']:30}")
    if not v["valido"]:
        print(f"            {v['valores_orfaos']} valores sem correspondência")
        if v["amostra_orfaos"] is not None:
            print("  Amostra de valores órfãos:")
            v["amostra_orfaos"].show(10, truncate=False)

# 6. VALORES AUSENTES (top 10 piores)
print("\n6. VALORES AUSENTES (Top 10 colunas com mais dados faltantes)")
print("-" * 80)
ausentes_ordenados = sorted(
    validacoes["valores_ausentes"], 
    key=lambda x: x["percentual_ausente"], 
    reverse=True
)[:10]

if any(x["percentual_ausente"] > 0 for x in ausentes_ordenados):
    print(f"{'Tabela':40} | {'Coluna':30} | {'Ausentes':>10} | {'%':>6} | {'Completude':>10}")
    print("-" * 105)
    for v in ausentes_ordenados:
        if v["percentual_ausente"] > 0:
            print(f"{v['tabela']:40} | {v['coluna']:30} | {v['valores_ausentes']:>10,} | {v['percentual_ausente']:>5.1f}% | {v['completude']:>9.1f}%")
else:
    print("✓ Não há valores ausentes em nenhuma coluna")

# ESTATÍSTICAS GERAIS
print("\n" + "="*80)
print("ESTATÍSTICAS GERAIS DE QUALIDADE")
print("="*80)

total_duplicados = sum(v["duplicados"] for v in validacoes["duplicados"])
total_intervalos_ok = sum(1 for v in validacoes["intervalos"] if v["valido"])
total_intervalos = len(validacoes["intervalos"])
total_codigos_ok = sum(1 for v in validacoes["codigos"] if v["valido"])
total_codigos = len(validacoes["codigos"])
total_anos_ok = sum(1 for v in validacoes["anos"] if v["valido"])
total_anos = len(validacoes["anos"])
total_integridade_ok = sum(1 for v in validacoes["integridade"] if v["valido"])
total_integridade = len(validacoes["integridade"])

print(f"\nDuplicados encontrados: {total_duplicados:,}")
print(f"Intervalos válidos: {total_intervalos_ok}/{total_intervalos} ({round(total_intervalos_ok/total_intervalos*100, 1)}%)")
print(f"Códigos UF válidos: {total_codigos_ok}/{total_codigos} ({round(total_codigos_ok/total_codigos*100, 1)}%)")
print(f"Anos válidos: {total_anos_ok}/{total_anos} ({round(total_anos_ok/total_anos*100, 1)}%)")
print(f"Integridade referencial: {total_integridade_ok}/{total_integridade} ({round(total_integridade_ok/total_integridade*100, 1)}%)")

# CONCLUSÃO
print("\n" + "="*80)
problemas_criticos = (
    total_duplicados > 0 or
    total_intervalos_ok < total_intervalos or
    total_codigos_ok < total_codigos or
    total_anos_ok < total_anos or
    total_integridade_ok < total_integridade
)

if problemas_criticos:
    print("⚠ ATENÇÃO: Problemas de qualidade detectados")
    print("Revise os relatórios acima antes de prosseguir para a próxima camada.")
else:
    print("✓ QUALIDADE OK: Dados prontos para a camada Silver")
    print("Todas as validações passaram com sucesso.")

print("="*80)


RESUMO DAS VALIDAÇÕES DE QUALIDADE

1. DUPLICADOS
--------------------------------------------------------------------------------
✓ OK            | indicador_alfabetizacao_uf               | 0 duplicados (0.0%)
✓ OK            | indicador_alfabetizacao_municipio        | 0 duplicados (0.0%)
✓ OK            | meta_alfabetizacao_brasil                | 0 duplicados (0.0%)
✓ OK            | meta_alfabetizacao_uf                    | 0 duplicados (0.0%)
✓ OK            | meta_alfabetizacao_municipio             | 0 duplicados (0.0%)
✗ ENCONTRADOS   | microdados_alunos                        | 1,515,671 duplicados (39.18%)
  Amostra dos duplicados:
+--------+-----+
|id_aluno|count|
+--------+-----+
|21050727|2    |
|26038625|2    |
|27024292|2    |
|31026285|2    |
|33000377|2    |
+--------+-----+
only showing top 5 rows
✓ OK            | resultados_e_metas_ufs_2025              | 0 duplicados (0.0%)

2. INTERVALOS NUMÉRICOS
---------------------------------------------------------------

In [0]:
# Salva os relatórios de validação como tabelas no Unity Catalog para auditoria
from datetime import datetime

print("\nSalvando relatórios de validação no Unity Catalog...")

# Adiciona timestamp a todos os relatórios
timestamp_validacao = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# 1. Relatório de Duplicados
df_duplicados = spark.createDataFrame([
    {
        "timestamp_validacao": timestamp_validacao,
        **{k: v for k, v in item.items() if k != "amostra"}
    }
    for item in validacoes["duplicados"]
])
salvar_tabela_silver(df_duplicados, "validacao_duplicados")

# 2. Relatório de Intervalos
df_intervalos = spark.createDataFrame([
    {
        "timestamp_validacao": timestamp_validacao,
        **item
    }
    for item in validacoes["intervalos"]
])
salvar_tabela_silver(df_intervalos, "validacao_intervalos")

# 3. Relatório de Códigos UF
df_codigos = spark.createDataFrame([
    {
        "timestamp_validacao": timestamp_validacao,
        **{k: v for k, v in item.items() if k != "amostra_invalidos"}
    }
    for item in validacoes["codigos"]
])
salvar_tabela_silver(df_codigos, "validacao_codigos_uf")

# 4. Relatório de Anos
df_anos = spark.createDataFrame([
    {
        "timestamp_validacao": timestamp_validacao,
        "tabela": item["tabela"],
        "coluna": item["coluna"],
        "total_registros": item["total_registros"],
        "anos_invalidos": item["anos_invalidos"],
        "anos_esperados": str(item["anos_esperados"]),
        "percentual_invalido": item["percentual_invalido"],
        "valido": item["valido"]
    }
    for item in validacoes["anos"]
])
salvar_tabela_silver(df_anos, "validacao_anos")

# 5. Relatório de Integridade Referencial
df_integridade = spark.createDataFrame([
    {
        "timestamp_validacao": timestamp_validacao,
        **{k: v for k, v in item.items() if k != "amostra_orfaos"}
    }
    for item in validacoes["integridade"]
])
salvar_tabela_silver(df_integridade, "validacao_integridade_referencial")

# 6. Relatório de Valores Ausentes
df_valores_ausentes = spark.createDataFrame([
    {
        "timestamp_validacao": timestamp_validacao,
        **item
    }
    for item in validacoes["valores_ausentes"]
])
salvar_tabela_silver(df_valores_ausentes, "validacao_valores_ausentes")

print("\n✓ Relatórios de validação salvos com sucesso!")
print("\nTabelas criadas:")
print("  - workspace.silver.validacao_duplicados")
print("  - workspace.silver.validacao_intervalos")
print("  - workspace.silver.validacao_codigos_uf")
print("  - workspace.silver.validacao_anos")
print("  - workspace.silver.validacao_integridade_referencial")
print("  - workspace.silver.validacao_valores_ausentes")


Salvando relatórios de validação no Unity Catalog...
✓ Tabela Silver criada: workspace.silver.validacao_duplicados (7 registros)
✓ Tabela Silver criada: workspace.silver.validacao_intervalos (21 registros)
✓ Tabela Silver criada: workspace.silver.validacao_codigos_uf (3 registros)
✓ Tabela Silver criada: workspace.silver.validacao_anos (7 registros)
✓ Tabela Silver criada: workspace.silver.validacao_integridade_referencial (3 registros)
✓ Tabela Silver criada: workspace.silver.validacao_valores_ausentes (97 registros)

✓ Relatórios de validação salvos com sucesso!

Tabelas criadas:
  - workspace.silver.validacao_duplicados
  - workspace.silver.validacao_intervalos
  - workspace.silver.validacao_codigos_uf
  - workspace.silver.validacao_anos
  - workspace.silver.validacao_integridade_referencial
  - workspace.silver.validacao_valores_ausentes


In [0]:
# Salvamento das bases silver tratadas como tabelas Delta no Unity Catalog
print("\n" + "="*80)
print("SALVANDO BASES SILVER TRATADAS")
print("="*80)
bases_silver = {
    "indicador_alfabetizacao_uf": indicador_uf,
    "indicador_alfabetizacao_municipio": indicador_municipio,
    "meta_alfabetizacao_brasil": meta_brasil,
    "meta_alfabetizacao_uf": meta_uf,
    "meta_alfabetizacao_municipio": meta_municipio,
    "microdados_alunos": microdados_alunos,
    "resultados_e_metas_ufs_2025": resultados_uf_2025,
}

for nome, df in bases_silver.items():
    salvar_tabela_silver(df, nome)


SALVANDO BASES SILVER TRATADAS
✓ Tabela Silver criada: workspace.silver.indicador_alfabetizacao_uf (145 registros)
✓ Tabela Silver criada: workspace.silver.indicador_alfabetizacao_municipio (23,995 registros)
✓ Tabela Silver criada: workspace.silver.meta_alfabetizacao_brasil (3 registros)
✓ Tabela Silver criada: workspace.silver.meta_alfabetizacao_uf (54 registros)
✓ Tabela Silver criada: workspace.silver.meta_alfabetizacao_municipio (10,704 registros)
✓ Tabela Silver criada: workspace.silver.microdados_alunos (3,867,999 registros)
✓ Tabela Silver criada: workspace.silver.resultados_e_metas_ufs_2025 (27 registros)


In [0]:
# Amostras visuais para comprovar as correções da Silver.
# A ideia aqui não é gerar relatório; é mostrar exemplos antes/depois.


def mostrar_amostra(titulo, df, linhas=10):
    print(f"\n{titulo}")
    try:
        display(df.limit(linhas))
    except NameError:
        df.show(linhas, truncate=False)


amostra_rede_uf = indicador_uf_raw.select(
    limpar_texto("ano").alias("ano_original"),
    limpar_texto("sigla_uf").alias("sigla_uf_original"),
    limpar_texto("rede").alias("rede_original"),
    normalizar_rede("rede").alias("rede_corrigida"),
).dropDuplicates(["rede_original", "rede_corrigida"])

amostra_tipos_meta = meta_uf_raw.select(
    limpar_texto("ano").alias("ano_original"),
    para_inteiro("ano").alias("ano_corrigido"),
    limpar_texto("taxa_alfabetizacao").alias("taxa_original"),
    para_decimal("taxa_alfabetizacao").alias("taxa_corrigida"),
    limpar_texto("meta_alfabetizacao_2030").alias("meta_2030_original"),
    para_decimal("meta_alfabetizacao_2030").alias("meta_2030_corrigida"),
).limit(20)

amostra_xlsx_limpo = resultados_uf_2025.select(
    "ano",
    "codigo_uf",
    "sigla_uf",
    "nome_uf",
    "rede",
    "resultado_2025",
    "meta_2025",
    "meta_2030",
    "percentual_participacao",
).orderBy("nome_uf")

amostra_microdados = microdados_raw.select(
    limpar_texto("ano").alias("ano_original"),
    para_inteiro("ano").alias("ano_corrigido"),
    limpar_texto("rede").alias("rede_original"),
    normalizar_rede("rede").alias("rede_corrigida"),
    limpar_texto("proficiencia").alias("proficiencia_original"),
    para_decimal("proficiencia").alias("proficiencia_corrigida"),
    limpar_texto("peso_aluno").alias("peso_aluno_original"),
    para_decimal("peso_aluno").alias("peso_aluno_corrigido"),
).limit(20)

amostra_nulos_indicador = indicador_municipio.select(
    "ano",
    "id_municipio",
    "rede",
    "taxa_alfabetizacao",
    "media_portugues",
    "proporcao_aluno_nivel_0",
    "proporcao_aluno_nivel_1",
).filter(F.col("proporcao_aluno_nivel_0").isNull()).limit(20)

mostrar_amostra("Amostra 1 - Normalização de rede por UF", amostra_rede_uf)
mostrar_amostra("Amostra 2 - Conversão de tipos e percentuais em metas UF", amostra_tipos_meta)
mostrar_amostra("Amostra 3 - XLSX limpo, com cabeçalhos extras removidos", amostra_xlsx_limpo)
mostrar_amostra("Amostra 4 - Microdados com rede e campos numéricos corrigidos", amostra_microdados)
mostrar_amostra("Amostra 5 - Valores ausentes preservados para validação", amostra_nulos_indicador)


Amostra 1 - Normalização de rede por UF


ano_original,sigla_uf_original,rede_original,rede_corrigida
2023,AM,3,municipal
2023,PB,2,estadual
2023,PR,5,publica
2024,BA,0,total



Amostra 2 - Conversão de tipos e percentuais em metas UF


ano_original,ano_corrigido,taxa_original,taxa_corrigida,meta_2030_original,meta_2030_corrigida
2024,2024,null,null,null,null
2023,2023,null,null,null,null
2023,2023,null,null,80,80.0
2024,2024,51.38,51.38,80,80.0
2023,2023,null,null,80,80.0
2024,2024,59.13,59.13,80,80.0
2023,2023,31.3,31.3,80,80.0
2024,2024,38.39,38.39,80,80.0
2024,2024,35.96,35.96,80,80.0
2023,2023,36.8,36.8,80,80.0



Amostra 3 - XLSX limpo, com cabeçalhos extras removidos


ano,codigo_uf,sigla_uf,nome_uf,rede,resultado_2025,meta_2025,meta_2030,percentual_participacao
2025,12,AC,Acre,publica,68.0,57.0,80.0,82.0
2025,27,AL,Alagoas,publica,64.0,56.0,80.0,94.0
2025,16,AP,Amapá,publica,60.0,54.0,80.0,88.0
2025,13,AM,Amazonas,publica,57.0,61.0,80.0,86.0
2025,29,BA,Bahia,publica,55.0,50.0,80.0,87.0
2025,23,CE,Ceará,publica,84.0,null,80.0,96.0
2025,53,DF,Distrito Federal,publica,65.0,63.0,80.0,85.0
2025,32,ES,Espírito Santo,publica,77.0,72.0,80.0,92.0
2025,52,GO,Goiás,publica,80.0,71.0,80.0,88.0
2025,21,MA,Maranhão,publica,69.0,64.0,80.0,92.0



Amostra 4 - Microdados com rede e campos numéricos corrigidos


ano_original,ano_corrigido,rede_original,rede_corrigida,proficiencia_original,proficiencia_corrigida,peso_aluno_original,peso_aluno_corrigido
2023,2023,3,municipal,null,null,null,null
2023,2023,3,municipal,null,null,null,null
2023,2023,3,municipal,null,null,null,null
2023,2023,3,municipal,null,null,null,null
2023,2023,3,municipal,null,null,null,null
2024,2024,3,municipal,null,null,null,null
2024,2024,3,municipal,null,null,null,null
2024,2024,3,municipal,null,null,null,null
2024,2024,3,municipal,null,null,null,null
2024,2024,3,municipal,null,null,null,null



Amostra 5 - Valores ausentes preservados para validação


ano,id_municipio,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1
2023,1100031,municipal,69.1,767.8763,null,null
2023,1100072,municipal,58.2,747.8918,null,null
2023,1100189,publica,69.73,762.4062,null,null
2023,1101609,municipal,50.7,745.6802,null,null
2023,1101807,municipal,55.69,752.3724,null,null
2023,1302900,municipal,53.17,731.0744,null,null
2023,1303304,estadual,72.32,755.225,null,null
2023,1500107,publica,39.73,725.5913,null,null
2023,1501105,publica,46.7,733.3467,null,null
2023,1501725,municipal,73.21,758.3553,null,null


In [0]:
# Salvamento das amostras como tabelas no Unity Catalog
amostras = {
    "amostra_02_rede_uf_antes_depois": amostra_rede_uf,
    "amostra_02_tipos_metas_uf_antes_depois": amostra_tipos_meta,
    "amostra_03_xlsx_resultados_uf_limpo": amostra_xlsx_limpo.limit(30),
    "amostra_04_microdados_antes_depois": amostra_microdados,
    "amostra_05_nulos_preservados_indicador_municipio": amostra_nulos_indicador,
}

for nome, df in amostras.items():
    salvar_tabela_silver(df, nome)

✓ Tabela Silver criada: workspace.silver.amostra_02_rede_uf_antes_depois (4 registros)
✓ Tabela Silver criada: workspace.silver.amostra_02_tipos_metas_uf_antes_depois (20 registros)
✓ Tabela Silver criada: workspace.silver.amostra_03_xlsx_resultados_uf_limpo (27 registros)
✓ Tabela Silver criada: workspace.silver.amostra_04_microdados_antes_depois (20 registros)
✓ Tabela Silver criada: workspace.silver.amostra_05_nulos_preservados_indicador_municipio (20 registros)
